# Illumination correction for spinning disk confocal microscopy

Illumination of an image isn't completely homogenous across the whole plane. Areas at the borders tend to be darker than the rest of the image (so-called *vignette effect*). In order to correct the intensity profile, your image must be normalised in relation to reference image of uniform fluorescence field (reference slide/chroma slide). This is the action performed by this script.

## INPUT:
- a file containing .nd2 images
- the script can handle higher dimensional images (meaning any combination of: [t,p,c,z,y,x])
- the script assumes all the input images have the same dimensionality, though it can handle most combinations with some exceptions;

- NOTE: if the input folder contains combination of images that do/don't contain z-stack, there is a problem when correcting with flatfield/darkfield. This issue wont be present with any other combination of dimensions (though some naming issues might arise, see below) 

## OUTPUT:
- file/-s containing corrected .tif images, with either [z,y,x] or [y,x] shape. Each channel is corrected individually as they require different reference (flatfield/darkfield) images. If the input file is is time-lapse/multipoint, then each instance will be separated into a new individual file.

- NOTE: if the input folder contains images that differ in dimensionality (e.g. both [t,c,z,y,x] and [c,z,y,x]) then there might be an issue with naming of the output files. The script assigns the same number of images from each channel to each input image name.

In [ ]:
## LIBRARY
from nd2 import ND2File
import nd2
from skimage import io
import numpy as np
from calmutils.imageio.tiff_imagej import save_tiff_imagej
import glob
from pathlib import Path
import warnings
import re

In [12]:
## PARAMETERS

# path to file containing images to correct
in_path = 'C:/Users/terez/Documents/Skola/Ludwig-Maximilians_Universitat_Munchen/Research_course_Leonhardt_lab/20240906_spinning_disk_dapi60x_for_basic'
# or subdirectory if images aren't directly in in_path
in_subdirectory = ''

# puts the corrected images in a subdirectory:
# NOTE: take care not to put any slashes into out_subdirectory, as that results in error (or the
#       images can end up in the root directory - e.g. directly on the C:/ drive)
out_subdirectory = 'corrected_images_file'

# naming preferences:
# enter your prefered name for a channel in the second column:
channel_renaming_dict = {
    '405' : '405',
    '488' : '488',
    '561' : '561',
    '640' : '640',
}
# e.g. if the img_suffix is set to 'ch_{channel_name}' (channel_name = name for the channel from
# channel_renaming_dict, e.g. 488-CSU-W1) the resulting name will be:
# name-of-original-image_corr_ch_488-CSU-W1.tif'
img_suffix = 'ch_{channel_name}'

# paths to the correct flatfield images
flatfield_paths_100x = {
    '405' : './images_miscelaneous/230824_flatfield_from_BaSiC_ch405.tif',
    '488' : './images_miscelaneous/120824_flatfield_img_488.tif',
    '561' : './images_miscelaneous/120824_flatfield_img_561.tif',
    '640' : './images_miscelaneous/120824_flatfield_img_640.tif'
} 
flatfield_paths_60x = {
    '405' : './images_miscelaneous/flatfields_darkfields/060924_flatfield_405_60x_from_BaSiC.tif',
    '488' : './images_miscelaneous/flatfields_darkfields/030924_flatfield_img_60x_488.tif',
    '561' : './images_miscelaneous/flatfields_darkfields/030924_flatfield_img_60x_561.tif',
    '640' : './images_miscelaneous/flatfields_darkfields/030924_flatfield_img_60x_640.tif'
}
flatfield_paths_40x = {
    '405' : './images_miscelaneous/290824_flatfield_405_40x_from_BaSiC.tif',
    '488' : './images_miscelaneous/120824_flatfield_img__40x_488.tif',
    '561' : './images_miscelaneous/120824_flatfield_img_40x_561.tif',
    '640' : './images_miscelaneous/120824_flatfield_img_40x_640.tif'
} 

# paths to the correct darkfield images
darkfield_paths = {
    '405' : './images_miscelaneous/230824_darkfield_from_BaSiC_ch405.tif',
    '488' : './images_miscelaneous/20240510-dark_img_avg-488.tif',
    '561' : './images_miscelaneous/20240510-dark_img_avg-640.tif', ##!!!! this is DF for 640
    '640' : './images_miscelaneous/20240510-dark_img_avg-640.tif'
} 

# pick flatfields:
flatfield_paths = flatfield_paths_100x

# darkfield fixed value
# if used the darkfield image will be disregarded and fixed value will be used
# values in a darkfield are usually around 490, but check through imageJ
use_fixed_darkfield_value = False
# darkfield_value = 490

darkfield_value_dict = {
    '405' : 490,
    '488' : 490,
    '561' : 490,
    '640' : 490
}


In [ ]:
## LOADING
files_to_correct = glob.glob(f"{in_path}{in_subdirectory}/*.nd2")
# since there's been issues
print(f'Number of files in given folder: {len(files_to_correct)}')

images_to_correct = {}
images_to_correct_ch405 = []
images_to_correct_ch488 = []
images_to_correct_ch561 = []
images_to_correct_ch640 = []

# function that picks which channel/s to correct
def pick_a_channel(reader, index):
    channel_select = []
    for key in reader.sizes.keys():
        if key == 'C':
            # for channel selection
            channel_select.append(index)
        else:
            # for everything else
            channel_select.append(slice(None))
    img_reduced_dim = reader.asarray().astype(float)
    img_to_correct = img_reduced_dim[tuple(channel_select)]
    return img_to_correct

# function that reduces multidimensional array into a list of images with shape z-y-x or y-x
def multidimensional_reduction(reader, image):
    img_to_correct_list = []
    if 'Z' in reader.sizes.keys():
        if len(image.shape) == 4:                  # shape: p, z, y, x (resp. t, z , y , x)
            for p in range(image.shape[0]):
                img_to_correct_list.append(image[p,:,:,:])
            return img_to_correct_list
        elif len(image.shape) == 5:                # shape: t, p, z, y, x
            for t in range(image.shape[0]):
                for p in range(image.shape[1]):
                    img_to_correct_list.append(image[t,p,:,:,:])
            return img_to_correct_list
        else:                                               # shape: z, y, x
            return image
    else:
        if len(image.shape) == 3:                  # shape: p, y, x (resp. t, y, x)
            for p in range(image.shape[0]):
                img_to_correct_list.append(image[p,:,:])
            return img_to_correct_list
        elif len(image.shape) == 4:                # shape: t, p, y, x
            for t in range(image.shape[0]):
                for p in range(image.shape[1]):
                    img_to_correct_list.append(image[t,p,:,:])
            return img_to_correct_list
        else:                                               # shape: y, x
            return image

def get_wavelengths(reader):
    # extraction of wavelength from text_info:
    pattern = r"(?:^|\n) +Line:\d;.*?(?=\n *Line:\d;|\n *(?=\S)|$)"
    matches = re.findall(pattern, reader.text_info['description'], re.DOTALL | re.MULTILINE)
    wavelengths = []
    for match in matches:
        if 'On' in match:
            key = re.search(r'Line:(\d+);', match).group(1)
            wavelength = re.search(r'ExW:(\d+);', match).group(1)
            wavelengths.append(wavelength)
    return wavelengths

# goes over every file and every channel and puts the channels into dictionary keyed by their 
# excitation wavelength
for filename in files_to_correct:
    with ND2File(filename) as reader:
        # pixel size
        pixel_size = reader.voxel_size()[::-1]
        # wavelengths
        wavelengths = get_wavelengths(reader)

        for i in range(len(wavelengths)):
            if wavelengths[i] == '405':
                image_to_corr = pick_a_channel(reader, i)
                images_to_correct_ch405.append(image_to_corr)
                images_to_correct[wavelengths[i]] = images_to_correct_ch405

            elif wavelengths[i] == '488':
                image_to_corr = pick_a_channel(reader, i)
                images_to_correct_ch488.append(image_to_corr)
                images_to_correct[wavelengths[i]] = images_to_correct_ch488

            elif wavelengths[i] == '561':
                image_to_corr = pick_a_channel(reader, i)
                images_to_correct_ch561.append(image_to_corr)
                images_to_correct[wavelengths[i]] = images_to_correct_ch561

            elif wavelengths[i] == '640':
                image_to_corr = pick_a_channel(reader, i)
                images_to_correct_ch640.append(image_to_corr)
                images_to_correct[wavelengths[i]] = images_to_correct_ch640
            else:
                warnings.warn("One or more of the channels aren't loaded")

        for key, value in images_to_correct.items():
            new_images_to_correct = []
            for v in range(len(value)):
                new_images_to_correct.append(multidimensional_reduction(reader, value[v]))
            while any(isinstance(item, list) for item in new_images_to_correct):
                new_images_to_correct = [item for sublist in new_images_to_correct for item in sublist]
            images_to_correct[key] = new_images_to_correct

# to check results
for key, value in images_to_correct.items():
    print(f'Channel: {key},\t number of images per channel: {len(value)}')

In [ ]:
## FLATFIELDS/DARKFIELDS LOADING AND RESHAPING
# flatfield/darkfield images:

# Creating dictionary for flatfields/darkfields: key: [flatfield_array, darkfield_array]
flatfields_darkfields = {}

for wave in wavelengths:
    flatfield = io.imread(flatfield_paths[wave]).astype(float)
    if use_fixed_darkfield_value == False:
        darkfield = io.imread(darkfield_paths[wave]).astype(float)
    else:
        darkfield = darkfield_value_dict[wave]
    flatfields_darkfields[wave] = [flatfield, darkfield]

# Reshaping flatfield array to match image_to_correct array:
first_key = next(iter(images_to_correct))

match_dims = []
# TODO: might fail in veeeerrry unlikely cases, e.g. 1024 multipoint positions
for dim in range(images_to_correct[first_key][0].ndim):
    if images_to_correct[first_key][0].shape[dim] in flatfields_darkfields[first_key][0].shape:
        match_dims.append(images_to_correct[first_key][0].shape[dim])
    else:
        match_dims.append(1)

for key,value in flatfields_darkfields.items():
    flatfields_reshaped = value[0].reshape(match_dims)
    if use_fixed_darkfield_value == False:
        darkfield_reshaped = value[1].reshape(match_dims)
    else:
        darkfield_reshaped = value[1]
    flatfields_darkfields[key] = [flatfields_reshaped, darkfield_reshaped]

# to check:
for wave in wavelengths:
    print(flatfields_darkfields[wave][0].shape)

#NOTE: the darkfield value for ch561 is still missing and therefore is replaced by darkfield
# of channel 640 in the dictionary (490) - correct later!


In [ ]:
## IMAGE CORRECTION
for key, value in images_to_correct.items():
    corrected_images = []
    for image in images_to_correct[key]:
        image = image-flatfields_darkfields[key][1]
        image_corr = image/flatfields_darkfields[key][0]
        corrected_images.append(image_corr)

    # Creating out_path:
    in_path = Path(in_path)
    out_path = in_path / out_subdirectory / f'channel_{key}'
    if not out_path.exists():
        out_path.mkdir(parents=True)
    # to check, since there have been problems:
    print(out_path)

## SAVE:
    k = 0
    nb = len(corrected_images)/len(files_to_correct)
    channel_name = channel_renaming_dict[key]
    for i in range(len(corrected_images)):
        if len(files_to_correct) != len(corrected_images):
            if i >= (k+1)*nb:
                k += 1
            name_img = Path(files_to_correct[k]).stem + f'_{i}_corr_{img_suffix.format(channel_name=channel_name)}.tif'
        else:
            name_img = Path(files_to_correct[i]).stem + f'_corr_{img_suffix.format(channel_name=channel_name)}.tif'
        print(name_img)

        save_tiff_imagej(f'{out_path}/{name_img}',
                    corrected_images[i].astype(np.float32), axes='zyx',
                    pixel_size=pixel_size, distance_unit='micron')
